# 🚀 MobileViT-S Edge Optimization & Quantization Benchmark
### Automated Waste Segregation System — Phase 2: Edge Deployment Validation
**Author:** Eshan Agarwal (Reg. No. 2430010407, Manipal University Jaipur)  
**Target:** Edge Devices (NVIDIA Jetson, Raspberry Pi 4/5, Industrial Edge Gateway)

---

### 📌 Objectives of this Notebook:
1. **Load Pre-Trained MobileViT-S:** Re-instantiate fine-tuned weights (`mobilevit_best.pth` with 96.06% accuracy).
2. **Export to ONNX (Open Neural Network Exchange):** Convert PyTorch graph to framework-agnostic ONNX runtime format (`mobilevit_s_fp32.onnx`).
3. **Post-Training Dynamic INT8 Quantization:** Quantize weights from 32-bit floating point to 8-bit integer (`mobilevit_s_int8.onnx`).
4. **Latency & Throughput Benchmarks:**
   - Single-item latency (Batch Size = 1) in milliseconds (critical for conveyor sorting).
   - Throughput (Batch Size = 32) in Frames Per Second (FPS).
   - Evaluated on **GPU (simulating Jetson)** and **CPU (simulating Raspberry Pi / edge host)**.
5. **Accuracy Verification:** Evaluate PyTorch FP32 vs ONNX FP32 vs ONNX INT8 on all **3,706 test images** to prove negligible accuracy loss (<0.5%).
6. **Generate Publication Tables & Figures:** Produce ready-to-paste IEEE LaTeX tables and comparison plots for the research paper.

## 1. Environment Setup & Hardware Inspection

In [ ]:
# ─── 1. Check Hardware ──────────────────────────────────
import torch
import os

print("="*60)
print("GPU Available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:      ", torch.cuda.get_device_name(0))
    print("GPU Memory:    ", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("ℹ️ Running in CPU mode. (For GPU benchmarks, enable T4 GPU in Runtime settings)")
print("="*60)

# ─── Install Required Edge Runtime Libraries ─────────
!pip install timm onnx onnxruntime onnxruntime-gpu kagglehub scikit-learn matplotlib seaborn tabulate --quiet
print("✅ All edge runtime packages installed successfully!")

## 2. Mount Google Drive & Locate Model Checkpoint

In [ ]:
# ─── Mount Google Drive ────────────────────────────────
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

# Potential locations (handling both 'MyDrive' and 'My Drive' with space)
CANDIDATE_PATHS = [
    '/content/drive/MyDrive/mobilevit_results/mobilevit_best.pth',
    '/content/drive/My Drive/mobilevit_results/mobilevit_best.pth',
    '/content/drive/MyDrive/mobilevit_best.pth',
    '/content/drive/My Drive/mobilevit_best.pth',
    '/content/mobilevit_best.pth'
]

CHECKPOINT_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        CHECKPOINT_PATH = p
        break

# Auto-search if not at the standard paths
if CHECKPOINT_PATH is None:
    print("🔍 Searching /content and Google Drive for 'mobilevit_best.pth'...")
    for root, dirs, files in os.walk('/content'):
        for f in files:
            if 'mobilevit' in f.lower() and f.endswith('.pth'):
                CHECKPOINT_PATH = os.path.join(root, f)
                break
        if CHECKPOINT_PATH:
            break

if CHECKPOINT_PATH is None:
    # List what exists in Drive to help diagnose
    drive_contents = []
    for d in ['/content/drive/MyDrive', '/content/drive/My Drive']:
        if os.path.exists(d):
            drive_contents.extend(os.listdir(d)[:15])
    raise FileNotFoundError(
        f"❌ Could not locate mobilevit_best.pth!
"
        f"Items found in your Drive root: {drive_contents}
"
        f"👉 Please check your Google Drive to see which folder it was saved to, or upload it to /content/."
    )

print(f"✅ Found checkpoint at: {CHECKPOINT_PATH}")
size_mb = os.path.getsize(CHECKPOINT_PATH) / (1024 * 1024)
print(f"📦 PyTorch Checkpoint File Size: {size_mb:.2f} MB")

## 3. Dataset Pipeline & Test Split Replication (Seed=42)
Recreating the exact 3,706-image test split used during baseline training.

In [ ]:
import kagglehub
import random
import numpy as np
import pandas as pd
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32

# Set reproducible seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("⏳ Downloading dataset from Kaggle...")
dataset_dir = kagglehub.dataset_download("sujaykapadnis/cnn-waste-classification-image-dataset")

DATASET_PATH = None
for root, dirs, files in os.walk(dataset_dir):
    dir_names = [d.upper() for d in dirs]
    if 'O' in dir_names and 'R' in dir_names:
        DATASET_PATH = root
        break
    if any('ORGANIC' in d for d in dir_names) and any('RECYCL' in d for d in dir_names):
        DATASET_PATH = root
        break

if DATASET_PATH is None:
    DATASET_PATH = dataset_dir

print("✅ Dataset root directory:", DATASET_PATH)

# Test Transform: Resize + Normalization (No data augmentation)
eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

full_dataset = ImageFolder(root=DATASET_PATH)
labels = [s[1] for s in full_dataset.samples]
indices = list(range(len(full_dataset)))

# Exact 70/15/15 stratified partition
idx_trainval, idx_test = train_test_split(
    indices, test_size=0.15, stratify=labels, random_state=SEED
)

class SplitDataset(Dataset):
    def __init__(self, dataset, indices, transform):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        img, label = self.dataset[self.indices[i]]
        return self.transform(img), label

test_ds = SplitDataset(full_dataset, idx_test, eval_transforms)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ Recreated Test Split: {len(test_ds):,} images ({full_dataset.classes})")

## 4. Load PyTorch Model & Baseline Test Evaluation

In [ ]:
import timm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
from tqdm import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Build model architecture
pytorch_model = timm.create_model('mobilevit_s', pretrained=False, num_classes=2)
state_dict = torch.load(CHECKPOINT_PATH, map_location='cpu')
pytorch_model.load_state_dict(state_dict)
pytorch_model.to(DEVICE)
pytorch_model.eval()

print("✅ MobileViT-S loaded successfully into memory.")
total_params = sum(p.numel() for p in pytorch_model.parameters())
print(f"📊 Total Parameters: {total_params:,}")

# Evaluate Baseline PyTorch Performance
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing PyTorch FP32"):
        images = images.to(DEVICE)
        outputs = pytorch_model(images)
        probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs)

pt_acc = accuracy_score(all_labels, all_preds) * 100
pt_f1 = f1_score(all_labels, all_preds) * 100
pt_auc = roc_auc_score(all_labels, all_probs)
pt_size = os.path.getsize(CHECKPOINT_PATH) / (1024 * 1024)

print("
" + "="*50)
print(f"  PyTorch FP32 Accuracy: {pt_acc:.2f}%")
print(f"  PyTorch FP32 F1-Score: {pt_f1:.2f}%")
print(f"  PyTorch FP32 ROC-AUC:  {pt_auc:.4f}")
print(f"  PyTorch FP32 Size:     {pt_size:.2f} MB")
print("="*50)

## 5. Export to ONNX (Open Neural Network Exchange)
Exporting the computational graph with dynamic batch dimension.

In [ ]:
import onnx

os.makedirs('/content/edge_models', exist_ok=True)
ONNX_FP32_PATH = '/content/edge_models/mobilevit_s_fp32.onnx'

# Switch to CPU for export
pytorch_model.eval().to('cpu')
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, requires_grad=False)

print("⏳ Exporting PyTorch graph to ONNX...")
torch.onnx.export(
    pytorch_model,
    dummy_input,
    ONNX_FP32_PATH,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

# Verify valid ONNX structure
onnx_model = onnx.load(ONNX_FP32_PATH)
onnx.checker.check_model(onnx_model)

onnx_fp32_size = os.path.getsize(ONNX_FP32_PATH) / (1024 * 1024)
print(f"✅ ONNX FP32 Model successfully exported!")
print(f"📁 Path: {ONNX_FP32_PATH}")
print(f"📦 ONNX FP32 File Size: {onnx_fp32_size:.2f} MB")

## 6. Post-Training Dynamic INT8 Quantization
Quantizes weights to 8-bit integers while preserving activation range.

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

ONNX_INT8_PATH = '/content/edge_models/mobilevit_s_int8.onnx'

print("⏳ Running Post-Training INT8 Dynamic Quantization...")
quantize_dynamic(
    model_input=ONNX_FP32_PATH,
    model_output=ONNX_INT8_PATH,
    weight_type=QuantType.QInt8
)

onnx_int8_size = os.path.getsize(ONNX_INT8_PATH) / (1024 * 1024)
reduction_pct = ((pt_size - onnx_int8_size) / pt_size) * 100

print(f"✅ Quantization Completed!")
print(f"📁 Path: {ONNX_INT8_PATH}")
print(f"📦 ONNX INT8 File Size:    {onnx_int8_size:.2f} MB")
print(f"📉 Storage Size Reduction: {reduction_pct:.2f}% (Compression Factor: {pt_size/onnx_int8_size:.1f}x)")

## 7. Verify Accuracy Retention (PyTorch vs ONNX FP32 vs ONNX INT8)
Evaluating on all 3,706 images in the test set to ensure INT8 quantization preserves accuracy.

In [ ]:
import onnxruntime as ort
from scipy.special import softmax

def evaluate_onnx_model(onnx_path, test_loader, name="ONNX"):
    session = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    input_name = session.get_inputs()[0].name
    
    preds, targets, probs = [], [], []
    for images, labels in tqdm(test_loader, desc=f"Evaluating {name}"):
        numpy_images = images.numpy()
        ort_outputs = session.run(None, {input_name: numpy_images})[0]
        prob_dist = softmax(ort_outputs, axis=1)
        
        preds.extend(np.argmax(prob_dist, axis=1))
        targets.extend(labels.numpy())
        probs.extend(prob_dist[:, 1])
        
    acc = accuracy_score(targets, preds) * 100
    f1 = f1_score(targets, preds) * 100
    auc = roc_auc_score(targets, probs)
    return acc, f1, auc

print("🔍 Evaluating ONNX FP32 Test Accuracy...")
onnx_fp32_acc, onnx_fp32_f1, onnx_fp32_auc = evaluate_onnx_model(ONNX_FP32_PATH, test_loader, name="ONNX FP32")

print("
🔍 Evaluating ONNX INT8 Test Accuracy...")
onnx_int8_acc, onnx_int8_f1, onnx_int8_auc = evaluate_onnx_model(ONNX_INT8_PATH, test_loader, name="ONNX INT8")

print("
" + "="*65)
print(f"{'Model Format':<18} | {'Accuracy (%)':<14} | {'F1-Score (%)':<14} | {'ROC-AUC':<10}")
print("-"*65)
print(f"{'PyTorch FP32':<18} | {pt_acc:<14.2f} | {pt_f1:<14.2f} | {pt_auc:<10.4f}")
print(f"{'ONNX FP32':<18} | {onnx_fp32_acc:<14.2f} | {onnx_fp32_f1:<14.2f} | {onnx_fp32_auc:<10.4f}")
print(f"{'ONNX INT8':<18} | {onnx_int8_acc:<14.2f} | {onnx_int8_f1:<14.2f} | {onnx_int8_auc:<10.4f}")
print("="*65)
print(f"Δ Accuracy (FP32 ➔ INT8): {onnx_int8_acc - pt_acc:+.2f}%")

## 8. Latency & Throughput Benchmarking (Edge Simulation)
Measuring single-item latency (Batch=1) and conveyor stream throughput (Batch=32).

In [ ]:
import time

def benchmark_pytorch(model, device, batch_size=1, runs=100, warmup=20):
    model.to(device)
    dummy = torch.randn(batch_size, 3, IMG_SIZE, IMG_SIZE).to(device)
    
    # Warmup
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(dummy)
        if device.type == 'cuda':
            torch.cuda.synchronize()
            
    latencies = []
    with torch.no_grad():
        for _ in range(runs):
            t0 = time.perf_counter()
            _ = model(dummy)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t1 = time.perf_counter()
            latencies.append((t1 - t0) * 1000.0) # in ms
            
    latencies = np.array(latencies)
    mean_lat = np.mean(latencies)
    p95_lat = np.percentile(latencies, 95)
    fps = (batch_size / (mean_lat / 1000.0))
    return mean_lat, p95_lat, fps

def benchmark_onnx(onnx_path, provider='CPUExecutionProvider', batch_size=1, runs=100, warmup=20):
    session = ort.InferenceSession(onnx_path, providers=[provider])
    input_name = session.get_inputs()[0].name
    dummy = np.random.randn(batch_size, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
    
    # Warmup
    for _ in range(warmup):
        _ = session.run(None, {input_name: dummy})
        
    latencies = []
    for _ in range(runs):
        t0 = time.perf_counter()
        _ = session.run(None, {input_name: dummy})
        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1000.0)
        
    latencies = np.array(latencies)
    mean_lat = np.mean(latencies)
    p95_lat = np.percentile(latencies, 95)
    fps = (batch_size / (mean_lat / 1000.0))
    return mean_lat, p95_lat, fps

print("⏱️ Benchmarking Latencies (Warmup=20, Iterations=100)...")

# 1. PyTorch on CPU
pt_cpu_b1_lat, pt_cpu_b1_p95, _ = benchmark_pytorch(pytorch_model, torch.device('cpu'), batch_size=1)
_, _, pt_cpu_b32_fps = benchmark_pytorch(pytorch_model, torch.device('cpu'), batch_size=32)

# 2. ONNX FP32 on CPU
onnx_fp32_b1_lat, onnx_fp32_b1_p95, _ = benchmark_onnx(ONNX_FP32_PATH, provider='CPUExecutionProvider', batch_size=1)
_, _, onnx_fp32_b32_fps = benchmark_onnx(ONNX_FP32_PATH, provider='CPUExecutionProvider', batch_size=32)

# 3. ONNX INT8 on CPU (Edge CPU Simulation)
onnx_int8_b1_lat, onnx_int8_b1_p95, _ = benchmark_onnx(ONNX_INT8_PATH, provider='CPUExecutionProvider', batch_size=1)
_, _, onnx_int8_b32_fps = benchmark_onnx(ONNX_INT8_PATH, provider='CPUExecutionProvider', batch_size=32)

# 4. GPU Benchmarks (if available)
has_gpu = torch.cuda.is_available()
if has_gpu:
    pt_gpu_b1_lat, pt_gpu_b1_p95, _ = benchmark_pytorch(pytorch_model, torch.device('cuda'), batch_size=1)
    _, _, pt_gpu_b32_fps = benchmark_pytorch(pytorch_model, torch.device('cuda'), batch_size=32)
else:
    pt_gpu_b1_lat, pt_gpu_b1_p95, pt_gpu_b32_fps = None, None, None

print("✅ Benchmarking complete!")

## 9. Results Compilation & IEEE LaTeX Table Generation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Compile results dictionary
results_data = [
    {
        "Engine": "PyTorch FP32",
        "Hardware": "Edge CPU (x86_64)",
        "Size (MB)": round(pt_size, 2),
        "Compression": "1.0x (Baseline)",
        "Latency B=1 (ms)": round(pt_cpu_b1_lat, 2),
        "P95 Latency (ms)": round(pt_cpu_b1_p95, 2),
        "Throughput B=32 (FPS)": round(pt_cpu_b32_fps, 1),
        "Accuracy (%)": round(pt_acc, 2),
        "F1 (%)": round(pt_f1, 2)
    },
    {
        "Engine": "ONNX Runtime FP32",
        "Hardware": "Edge CPU (x86_64)",
        "Size (MB)": round(onnx_fp32_size, 2),
        "Compression": f"{pt_size/onnx_fp32_size:.1f}x",
        "Latency B=1 (ms)": round(onnx_fp32_b1_lat, 2),
        "P95 Latency (ms)": round(onnx_fp32_b1_p95, 2),
        "Throughput B=32 (FPS)": round(onnx_fp32_b32_fps, 1),
        "Accuracy (%)": round(onnx_fp32_acc, 2),
        "F1 (%)": round(onnx_fp32_f1, 2)
    },
    {
        "Engine": "ONNX Runtime INT8",
        "Hardware": "Edge CPU (x86_64)",
        "Size (MB)": round(onnx_int8_size, 2),
        "Compression": f"{pt_size/onnx_int8_size:.1f}x ({reduction_pct:.1f}% reduction)",
        "Latency B=1 (ms)": round(onnx_int8_b1_lat, 2),
        "P95 Latency (ms)": round(onnx_int8_b1_p95, 2),
        "Throughput B=32 (FPS)": round(onnx_int8_b32_fps, 1),
        "Accuracy (%)": round(onnx_int8_acc, 2),
        "F1 (%)": round(onnx_int8_f1, 2)
    }
]

if has_gpu:
    results_data.append({
        "Engine": "PyTorch FP32",
        "Hardware": f"GPU ({torch.cuda.get_device_name(0)})",
        "Size (MB)": round(pt_size, 2),
        "Compression": "1.0x",
        "Latency B=1 (ms)": round(pt_gpu_b1_lat, 2),
        "P95 Latency (ms)": round(pt_gpu_b1_p95, 2),
        "Throughput B=32 (FPS)": round(pt_gpu_b32_fps, 1),
        "Accuracy (%)": round(pt_acc, 2),
        "F1 (%)": round(pt_f1, 2)
    })

df_results = pd.DataFrame(results_data)
print(df_results.to_markdown(index=False))

# Save results CSV
OUTPUT_DIR = '/content/drive/MyDrive/mobilevit_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
df_results.to_csv(f'{OUTPUT_DIR}/edge_quantization_benchmark.csv', index=False)
print(f"\n💾 Benchmark table saved to: {OUTPUT_DIR}/edge_quantization_benchmark.csv")

# ─── Plot Visual Comparison ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Model Size
engines = [r['Engine'] for r in results_data if 'Edge CPU' in r['Hardware']]
sizes = [r['Size (MB)'] for r in results_data if 'Edge CPU' in r['Hardware']]
axes[0].bar(engines, sizes, color=['#2b5c8f', '#4682b4', '#2e8b57'], width=0.55)
axes[0].set_title("Model Footprint (MB) — Lower is Better", fontsize=12, fontweight='bold')
axes[0].set_ylabel("Size on Disk (MB)")
for i, v in enumerate(sizes):
    axes[0].text(i, v + 0.3, f"{v} MB", ha='center', fontweight='bold')

# Plot 2: Single-item Latency (Batch=1)
latencies = [r['Latency B=1 (ms)'] for r in results_data if 'Edge CPU' in r['Hardware']]
axes[1].bar(engines, latencies, color=['#d9534f', '#f0ad4e', '#5cb85c'], width=0.55)
axes[1].set_title("Inference Latency @ Batch=1 (ms) — Lower is Better", fontsize=12, fontweight='bold')
axes[1].set_ylabel("Latency (ms)")
for i, v in enumerate(latencies):
    axes[1].text(i, v + 1.0, f"{v} ms", ha='center', fontweight='bold')

# Plot 3: Test Accuracy
accs = [r['Accuracy (%)'] for r in results_data if 'Edge CPU' in r['Hardware']]
axes[2].bar(engines, accs, color=['#6a5acd', '#483d8b', '#20b2aa'], width=0.55)
axes[2].set_title("Test Accuracy (%) — Higher is Better", fontsize=12, fontweight='bold')
axes[2].set_ylabel("Accuracy (%)")
axes[2].set_ylim(90, 100)
for i, v in enumerate(accs):
    axes[2].text(i, v + 0.3, f"{v}%", ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/edge_benchmark_comparison.png', dpi=300)
plt.show()

# ─── Formatted IEEE LaTeX Table ───────────────────────
print("\n" + "="*70)
print("             COPY-PASTE IEEE LaTeX TABLE FOR RESEARCH PAPER")
print("="*70)
latex_table = f"""
\\begin{{table}}[htbp]
\\centering
\\caption{{Edge Optimization and Dynamic Quantization Benchmark (MobileViT-S)}}
\\label{{tab:edge_quantization}}
\\resizebox{{\\columnwidth}}{{!}}{{
\\begin{{tabular}}{{lccccc}}
\\hline
\\textbf{{Inference Engine}} & \\textbf{{Precision}} & \\textbf{{Size (MB)}} & \\textbf{{Latency (ms)}} & \\textbf{{FPS}} & \\textbf{{Acc (\\%)}} \\ \\
\\hline
PyTorch (TorchScript)       & FP32 & {pt_size:.2f} & {pt_cpu_b1_lat:.2f} & {pt_cpu_b32_fps:.1f} & {pt_acc:.2f} \\
ONNX Runtime                 & FP32 & {onnx_fp32_size:.2f} & {onnx_fp32_b1_lat:.2f} & {onnx_fp32_b32_fps:.1f} & {onnx_fp32_acc:.2f} \\
\\textbf{{ONNX Dynamic Quant}} & \\textbf{{INT8}} & \\textbf{{{onnx_int8_size:.2f}}} & \\textbf{{{onnx_int8_b1_lat:.2f}}} & \\textbf{{{onnx_int8_b32_fps:.1f}}} & \\textbf{{{onnx_int8_acc:.2f}}} \\
\\hline
\\end{{tabular}}
}}
\\end{{table}}
"""
print(latex_table)

## 10. Persist Optimized ONNX Models to Google Drive

In [ ]:
import shutil

print(f"⏳ Copying optimized models to {OUTPUT_DIR}...")
shutil.copy(ONNX_FP32_PATH, f'{OUTPUT_DIR}/mobilevit_s_fp32.onnx')
shutil.copy(ONNX_INT8_PATH, f'{OUTPUT_DIR}/mobilevit_s_int8.onnx')

print("✅ Saved files:")
print(f"   1. {OUTPUT_DIR}/mobilevit_s_fp32.onnx")
print(f"   2. {OUTPUT_DIR}/mobilevit_s_int8.onnx")
print(f"   3. {OUTPUT_DIR}/edge_quantization_benchmark.csv")
print(f"   4. {OUTPUT_DIR}/edge_benchmark_comparison.png")
print("\n🎉 Phase 2 Edge Quantization & Benchmarking Successfully Completed!")